In [16]:
import json
import uuid
from typing import Annotated, Literal, Sequence, TypedDict
from langchain_core.messages import (
    AIMessage,
    BaseMessage,
    HumanMessage,
    ToolCall,
    ToolMessage,
)
from langgraph.graph.message import add_messages
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langgraph.graph import END, StateGraph
from langgraph.checkpoint.memory import MemorySaver
from langgraph.checkpoint.base import empty_checkpoint

In [11]:
class UserState(TypedDict):
    """
    State with messages for each session/user.
    """

    messages: Annotated[Sequence[BaseMessage], add_messages]

async def acall_model(state: UserState, config: RunnableConfig): 
    return {"messages": []}

async def tool_node(state: UserState, config: RunnableConfig):
    return {"messages": []}

async def booking_validation_node(state: UserState):
    return {"messages": []}

async def insert_ticket_node(state: UserState):
    return {"messages": []}

async def request_login_node(state: UserState):
    return {"messages": []}

def agent_should_continue(
    state: UserState, config: RunnableConfig
) -> Literal["booking_validation", "continue", "request_login", "end"]:
    return "end"

def booking_should_continue(state: UserState) -> Literal["continue", "agent"]:
    return "agent"    

AGENT_NODE = "agent"
TOOL_NODE = "tools"
BOOKING_VALIDATION_NODE = "booking_validation"
INSERT_TICKET_NODE = "insert_ticket"
REQUEST_LOGIN_NODE = "request_login"

llm_graph = StateGraph(UserState)
llm_graph.add_node(AGENT_NODE, RunnableLambda(acall_model))
llm_graph.add_node(TOOL_NODE, tool_node)
llm_graph.add_node(BOOKING_VALIDATION_NODE, RunnableLambda(booking_validation_node))
llm_graph.add_node(INSERT_TICKET_NODE, RunnableLambda(insert_ticket_node))
llm_graph.add_node(REQUEST_LOGIN_NODE, request_login_node)

llm_graph.set_entry_point(AGENT_NODE)

llm_graph.add_conditional_edges(
    AGENT_NODE,
    agent_should_continue,
    {
        "continue": TOOL_NODE,
        "booking_validation": BOOKING_VALIDATION_NODE,
        "request_login": REQUEST_LOGIN_NODE,
        "end": END,
    },
)
llm_graph.add_edge(TOOL_NODE, AGENT_NODE)
llm_graph.add_conditional_edges(
    BOOKING_VALIDATION_NODE,
    booking_should_continue,
    {"continue": INSERT_TICKET_NODE, "agent": AGENT_NODE},
)
llm_graph.add_edge(INSERT_TICKET_NODE, END)
llm_graph.add_edge(REQUEST_LOGIN_NODE, END)

checkpointer = MemorySaver()
langgraph_app = llm_graph.compile(
    checkpointer=checkpointer, 
    debug=False, 
    interrupt_after=["booking_validation"]
)

config = {
    "configurable": {
        "thread_id": "123asd",
        "checkpoint_ns": ""
    }
}

In [22]:
langgraph_app.get_state(config).values

{'messages': [HumanMessage(content='When is the next flight to Los Angeles?', additional_kwargs={}, response_metadata={}, id='c96814c2-8b64-443e-8f25-6cacc875884f')]}

In [13]:
langgraph_app.update_state(
    config, 
    {"messages": AIMessage(content="Welcome to Cymbal Air!  How may I assist you?")},
)

{'configurable': {'thread_id': '123asd',
  'checkpoint_ns': '',
  'checkpoint_id': '1f155c0d-8dfe-6c72-8000-9e95cddec593'}}

In [19]:
# Reset
checkpoint = empty_checkpoint()
checkpointer.put(
    config=config, 
    checkpoint=checkpoint, 
    metadata={"step": -1},  
    new_versions={}
)

{'configurable': {'thread_id': '123asd',
  'checkpoint_ns': '',
  'checkpoint_id': '1f155c0f-fb2b-6858-bffe-9b5bc3d9f30f'}}

In [21]:
# With user prompt
final_state = await langgraph_app.ainvoke(
    {"messages": HumanMessage(content="When is the next flight to Los Angeles?")},
    config=config,
)

In [23]:
final_state['messages']

[HumanMessage(content='When is the next flight to Los Angeles?', additional_kwargs={}, response_metadata={}, id='c96814c2-8b64-443e-8f25-6cacc875884f')]